# 01 · Trending topics de hoy con Selenium

**Máster Universitario en Ciberseguridad · Ciberinteligencia · Taller OSINT**

Recupera las 50 tendencias de X (Twitter) en España desde [getdaytrends.com](https://getdaytrends.com/es/spain/), las guarda en una tabla y conserva la evidencia de la captura.



> **Términos de uso.** getdaytrends permite copiar información puntual y hacer capturas, pero prohíbe la extracción y el almacenamiento masivos. Aquí consultamos **una página**, con fines académicos.

## 1 · Preparar el entorno

Instala Chrome en la máquina de Colab. Solo hay que hacerlo una vez por sesión.

In [2]:
# Instala Selenium y Google Chrome en la máquina de Colab (tarda 1-2 minutos)
!pip -q install selenium
!apt-get -qq update > /dev/null
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null
!rm -f google-chrome-stable_current_amd64.deb
!google-chrome --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Google Chrome 154.0.8037.57 


## 2 · Funciones

Definimos las piezas una vez y las reutilizamos. Lee los comentarios: cada función corresponde a una diapositiva de conceptos.

In [3]:
import selenium
from selenium import webdriver

def crear_driver():
    """Abre un Chrome sin ventana (headless), preparado para funcionar en Colab."""
    opciones = webdriver.ChromeOptions()
    for arg in ["--headless=new",          # sin ventana
                "--no-sandbox",            # necesario en Colab
                "--disable-dev-shm-usage", # evita problemas de memoria compartida en Colab
                "--disable-gpu",
                "--window-size=1920,1080", # tamaño de pantalla simulado (afecta a qué es 'visible')
                "--lang=es-ES"]:
        opciones.add_argument(arg)
    # Desde Selenium 4.6, Selenium Manager descarga automáticamente el chromedriver
    # que corresponde a la versión de Chrome instalada: ya no hace falta webdriver-manager.
    return webdriver.Chrome(options=opciones)

print("Selenium", selenium.__version__)

Selenium 4.49.0


In [4]:
import time, re, os, json, hashlib
from datetime import datetime, timezone, date, timedelta

import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (TimeoutException, WebDriverException,
                                        NoSuchElementException, ElementClickInterceptedException)

In [5]:
XPATH_TENDENCIAS = '//*[@id="trends"]//a[contains(@href, "/trend/")]'

def cargar_pagina(wd, url, reintentos=3, espera_max=15):
    """Abre la URL y espera (como máximo espera_max segundos) a que exista la tabla de tendencias.
    Reintenta si la web da error (getdaytrends devuelve a veces un 502)."""
    for intento in range(1, reintentos + 1):
        try:
            wd.get(url)
            if "502" in wd.title or "bad gateway" in wd.title.lower():
                raise WebDriverException("la web ha devuelto un error 502")
            WebDriverWait(wd, espera_max).until(
                EC.presence_of_element_located((By.ID, "trends")))
            print(f"✓ Cargada: {wd.title}")
            return True
        except (TimeoutException, WebDriverException) as e:
            motivo = (str(e).strip().splitlines() or [""])[0]
            print(f"✗ Intento {intento}/{reintentos}: {type(e).__name__} {motivo}")
            time.sleep(5)
    return False


def extraer_tendencias(wd, maximo=50):
    """Devuelve una lista de diccionarios con las tendencias de la página cargada.
    Usa textContent y no .text, porque las filas 16-50 están ocultas y .text las devolvería vacías."""
    filas, vistos = [], set()
    for a in wd.find_elements(By.XPATH, XPATH_TENDENCIAS):
        url = a.get_attribute("href")
        if url in vistos:          # por si la misma tendencia aparece enlazada dos veces
            continue
        vistos.add(url)
        nombre = (a.get_attribute("textContent") or "").strip()
        volumen = None             # p. ej. "57.9K tweets" o "Menos de 10K tweets", si la web lo muestra
        try:
            celda = a.find_element(By.XPATH, "./ancestor::td[1]")
            m = re.search(r"(Menos de [\d.,]+K|[\d.,]+K?) tweets", celda.get_attribute("textContent") or "")
            volumen = m.group(0) if m else None
        except NoSuchElementException:
            pass
        filas.append({"posicion": len(filas) + 1, "tendencia": nombre,
                      "volumen": volumen, "url_tendencia": url})
        if len(filas) == maximo:
            break
    return filas

In [6]:
def pulsar_ver_todo(wd):
    """Pulsa 'Ver todo 50'. No es imprescindible (leemos textContent), pero muestra cómo interactuar."""
    try:
        boton = WebDriverWait(wd, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href="#moreTrends"]')))
        wd.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton)  # scroll hasta el botón
        try:
            boton.click()
        except ElementClickInterceptedException:          # algo lo tapa (p. ej. un anuncio)
            wd.execute_script("arguments[0].click();", boton)
        print("✓ Pulsado 'Ver todo 50'")
    except TimeoutException:
        print("No se encuentra el botón 'Ver todo 50' (no pasa nada: leemos también las filas ocultas)")


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(65536), b""):
            h.update(bloque)
    return h.hexdigest()


def guardar_evidencia(wd, df, url, carpeta="evidencias", prefijo="getdaytrends"):
    """Guarda HTML, captura y CSV con la marca de tiempo UTC, y un manifiesto con el hash SHA-256 de cada fichero."""
    os.makedirs(carpeta, exist_ok=True)
    marca = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    base = os.path.join(carpeta, f"{prefijo}_{marca}")
    rutas = {"html": base + ".html", "captura": base + ".png", "csv": base + ".csv"}
    with open(rutas["html"], "w", encoding="utf-8") as f:
        f.write(wd.page_source)          # ojo: es el DOM tal como lo ve el navegador, no el HTML original
    wd.save_screenshot(rutas["captura"])
    df.to_csv(rutas["csv"], index=False, encoding="utf-8")
    manifiesto = {
        "url": url,
        "capturado_utc": marca,
        "user_agent": wd.execute_script("return navigator.userAgent"),
        "herramienta": f"Selenium {selenium.__version__}",
        "filas": len(df),
        "ficheros": {k: {"ruta": v, "sha256": sha256(v)} for k, v in rutas.items()},
    }
    with open(base + "_manifiesto.json", "w", encoding="utf-8") as f:
        json.dump(manifiesto, f, ensure_ascii=False, indent=2)
    return manifiesto

## 3 · Abrir el navegador y cargar la página

In [7]:
PAIS = "spain"
URL = f"https://getdaytrends.com/es/{PAIS}/"

wd = crear_driver()
ok = cargar_pagina(wd, URL)

✓ Cargada: España • Ahora • Tendencias de Twitter y Hashtags Populares


## 4 · Visible ≠ existe

Antes de pulsar nada, contamos cuántas tendencias hay en el HTML y cuántas devuelve `.text`.

In [8]:
enlaces = wd.find_elements(By.XPATH, XPATH_TENDENCIAS)
vacios = sum(1 for a in enlaces if a.text.strip() == "")
print(f"Enlaces a tendencias en la página: {len(enlaces)}")
print(f"De ellos, .text devuelve vacío en: {vacios}  ← son las filas ocultas")

Enlaces a tendencias en la página: 50
De ellos, .text devuelve vacío en: 35  ← son las filas ocultas


## 5 · Interactuar: pulsar «Ver todo 50»

In [9]:
pulsar_ver_todo(wd)

✓ Pulsado 'Ver todo 50'


## 6 · Extraer las tendencias y pasarlas a un DataFrame

In [10]:
filas = extraer_tendencias(wd, maximo=50)
df = pd.DataFrame(filas)
df["pais"] = PAIS
df["capturado_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
df["url_fuente"] = URL
print(f"✓ {len(df)} tendencias")
df.head(15)

✓ 50 tendencias


,posicion,tendencia,volumen,url_tendencia,pais,capturado_utc,url_fuente
0,1,Mari Carmen,None,https://getdaytrends.com/es/spain/trend/Mari%2...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
1,2,PSOE,None,https://getdaytrends.com/es/spain/trend/PSOE/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
2,3,Congreso,None,https://getdaytrends.com/es/spain/trend/Congreso/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
3,4,Retiro,None,https://getdaytrends.com/es/spain/trend/Retiro/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
4,5,Franco,None,https://getdaytrends.com/es/spain/trend/Franco/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
5,6,Ana Vázquez,None,https://getdaytrends.com/es/spain/trend/Ana%20...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
6,7,Ayuso,None,https://getdaytrends.com/es/spain/trend/Ayuso/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
7,8,#LaIslaDeLasTentaciones6,None,https://getdaytrends.com/es/spain/trend/%23LaI...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
8,9,Ana Belén,None,https://getdaytrends.com/es/spain/trend/Ana%20...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
9,10,Zapatero,None,https://getdaytrends.com/es/spain/trend/Zapatero/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/


### ✅ Checkpoint

Deberías ver una tabla con **50 filas**. Si tienes menos, mira el mensaje de error de las celdas anteriores.

## 7 · Guardar la evidencia

Mañana estas tendencias habrán cambiado. Guardamos la URL, la fecha y hora, una captura, la copia de la página y el contexto, con el hash SHA-256 de cada fichero para poder demostrar que no se han modificado.

In [11]:
manifiesto = guardar_evidencia(wd, df, URL)
print(json.dumps(manifiesto, ensure_ascii=False, indent=2))

{
  "url": "https://getdaytrends.com/es/spain/",
  "capturado_utc": "20260924T185700Z",
  "user_agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36",
  "herramienta": "Selenium 4.49.0",
  "filas": 50,
  "ficheros": {
    "html": {
      "ruta": "evidencias/getdaytrends_20260924T185700Z.html",
      "sha256": "bf1a4bffd4ee53e1b5965a7df8e7a040bc1d1e7a5c98b203c99c9eba5afceac6"
    },
    "captura": {
      "ruta": "evidencias/getdaytrends_20260924T185700Z.png",
      "sha256": "0230d4c2ccbaaa39ba40029083a223ddd99c5a1b0d29db34b0be3fc60bb4d87d"
    },
    "csv": {
      "ruta": "evidencias/getdaytrends_20260924T185700Z.csv",
      "sha256": "8b264eb96f2c59f534df82737e08f51a95b958832639f760c4704ee16743f65a"
    }
  }
}


In [12]:
# Para descargarlo todo a tu ordenador:
# !zip -qr evidencias.zip evidencias
# from google.colab import files; files.download("evidencias.zip")

## 8 · Otro país

La misma función sirve para cualquiera de los países de getdaytrends: `mexico`, `argentina`, `france`, `united-kingdom`, `united-states`… (el nombre es el que aparece en la URL).

In [13]:
def obtener_tt(wd, pais, maximo=50):
    url = f"https://getdaytrends.com/es/{pais}/"
    if not cargar_pagina(wd, url):
        return pd.DataFrame()
    d = pd.DataFrame(extraer_tendencias(wd, maximo))
    d["pais"] = pais
    d["capturado_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    return d

OTRO_PAIS = "mexico"
df_otro = obtener_tt(wd, OTRO_PAIS)
comunes = sorted(set(df["tendencia"].str.lower()) & set(df_otro["tendencia"].str.lower()))
print(f"Tendencias comunes entre {PAIS} y {OTRO_PAIS}: {comunes or 'ninguna'}")

✓ Cargada: Méjico • Ahora • Tendencias de Twitter y Hashtags Populares
Tendencias comunes entre spain y mexico: ['#felizjueves', 'franco', 'rockstar', 'xi jinping']


## 9 · Cerrar el navegador

In [14]:
wd.quit()
print("Navegador cerrado")

Navegador cerrado


## Plan B: si getdaytrends no responde

Antes de clase, ejecuta el notebook y descarga `evidencias.zip`. Si en clase la web falla, sube el CSV a Colab (icono de carpeta, a la izquierda) y cárgalo aquí:

In [15]:
import glob
respaldo = sorted(glob.glob("evidencias/*.csv") + glob.glob("*.csv"))
if respaldo:
    df = pd.read_csv(respaldo[-1])
    print(f"Cargado el respaldo {respaldo[-1]}: {len(df)} filas")
    display(df.head(15))
else:
    print("No hay ningún CSV de respaldo en la sesión")

Cargado el respaldo evidencias/getdaytrends_20260924T185700Z.csv: 50 filas


,posicion,tendencia,volumen,url_tendencia,pais,capturado_utc,url_fuente
0,1,Mari Carmen,NaN,https://getdaytrends.com/es/spain/trend/Mari%2...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
1,2,PSOE,NaN,https://getdaytrends.com/es/spain/trend/PSOE/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
2,3,Congreso,NaN,https://getdaytrends.com/es/spain/trend/Congreso/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
3,4,Retiro,NaN,https://getdaytrends.com/es/spain/trend/Retiro/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
4,5,Franco,NaN,https://getdaytrends.com/es/spain/trend/Franco/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
5,6,Ana Vázquez,NaN,https://getdaytrends.com/es/spain/trend/Ana%20...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
6,7,Ayuso,NaN,https://getdaytrends.com/es/spain/trend/Ayuso/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
7,8,#LaIslaDeLasTentaciones6,NaN,https://getdaytrends.com/es/spain/trend/%23LaI...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
8,9,Ana Belén,NaN,https://getdaytrends.com/es/spain/trend/Ana%20...,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
9,10,Zapatero,NaN,https://getdaytrends.com/es/spain/trend/Zapatero/,spain,2026-09-24T18:57:00+00:00,https://getdaytrends.com/es/spain/
